# 16 · Motion from keypoints

Motion from a video by its keypoints alone: the quick variant of the motion stage (notebook 06). Instead of SAM2 masks, the detector runs on every frame and each person is followed by matching their box to the frame's detections. ViTPose keypoints, MediaPipe hands and face blendshapes are found in each box as before, and the avatar's own Anny body is fitted to them frame by frame: root placement, every bone and finger, root motion. The silhouette phase is skipped.

It is faster and holds up on long videos, but it sees less: nothing tells a keypoint hidden behind someone from one of the person's own, nothing pulls the body's outline onto the person's, and people who cross each other may swap boxes (the track plot below shows it).

Run it from the shell (`go-task motion -- path/to/video.mp4 --keypoints-only`), or set `RUN` below. Results go to `outputs/motion/<name>/`, the `.motion` file the viewer plays among them.

Colours: one per person, in the order they were found.

In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Video
from PIL import Image

from avatar_splats.config import load_config
from avatar_splats.data import read_frames, write_video
from avatar_splats.export import AvatarRuntime, exported_path, read_avatar
from avatar_splats.motion import MotionClip, MotionFit, Observations, extract_motion, motion_path
from avatar_splats.motion.preview import clip_camera, render_clip

config = load_config("../configs/default.yaml")
device = config.motion.device

VIDEO = Path("../examples/striptease.mp4")  # the motion video
NAME = f"{VIDEO.stem}-keypoints"
RUN = True  # True: extract the motion here; False: read an earlier run
REFIT = False  # with RUN: fit again from the saved observations, skipping detection

if RUN:
    started = time.perf_counter()
    extract_motion(config, VIDEO, NAME, refit=REFIT, keypoints_only=True)
    print(f"extracted in {(time.perf_counter() - started) / 60:.1f} min")
observations = Observations.load(motion_path(config, NAME, "observations"))
fit = MotionFit.load(motion_path(config, NAME, "fit"))
clip = MotionClip.load(motion_path(config, NAME, "clip"))
frames = list(read_frames(VIDEO, step=config.motion.frame_step))
COLOURS = np.array([(230, 80, 60), (60, 140, 230), (90, 190, 90), (220, 180, 40)])
print(f"{observations.num_people} people, {len(frames)} frames at {observations.fps:.1f} fps")

## Who was found

Each person's box and keypoints on a few frames. A box that jumps to someone else shows up here first.

In [ ]:
shown = np.linspace(0, len(frames) - 1, 6).round().astype(int)
fig, axes = plt.subplots(1, len(shown), figsize=(20, 7))
for ax, f in zip(axes, shown, strict=True):
    ax.imshow(frames[f])
    for p in range(observations.num_people):
        colour = COLOURS[p % len(COLOURS)] / 255
        box = observations.boxes[p, f]
        if not np.isnan(box).any():
            x0, y0, x1, y1 = box
            ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, color=colour))
        points = observations.keypoints[p, f]
        seen = points[:, 2] >= 0.3
        ax.scatter(*points[seen, :2].T, s=8, color=colour, edgecolor="k")
    ax.set_title(f"frame {observations.frame_indices[f]}")
    ax.axis("off")
fig.tight_layout()

## The tracks

Where each person's box is across the video, and the frames they were missed in (grey). A swap shows as two tracks trading places at once; a lost person as a gap that the box is found again after.

In [ ]:
fig, (x_ax, size_ax) = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
for p in range(observations.num_people):
    colour = COLOURS[p % len(COLOURS)] / 255
    boxes = observations.boxes[p]
    x_ax.plot(observations.frame_indices, 0.5 * (boxes[:, 0] + boxes[:, 2]), color=colour)
    size_ax.plot(observations.frame_indices, boxes[:, 3] - boxes[:, 1], color=colour)
    for f in np.flatnonzero(np.isnan(boxes).any(axis=1)):
        x_ax.axvspan(
            observations.frame_indices[f] - 0.5, observations.frame_indices[f] + 0.5, color="0.9"
        )
x_ax.set_ylabel("box centre x (px)")
size_ax.set_ylabel("box height (px)")
size_ax.set_xlabel("frame")
fig.tight_layout()

## Per-frame fit

Keypoint error in pixels per person and frame. Grey bands mark frames where a person's keypoints were not found; their pose is only interpolated. There are no silhouettes to score.

In [ ]:
plt.figure(figsize=(12, 3))
for p in range(observations.num_people):
    plt.plot(
        fit.frame_indices,
        fit.keypoint_error[p],
        color=COLOURS[p % len(COLOURS)] / 255,
        label=f"person {p}",
    )
    for f in np.flatnonzero(~fit.seen[p]):
        plt.axvspan(fit.frame_indices[f] - 0.5, fit.frame_indices[f] + 0.5, color="0.9")
plt.ylabel("keypoint error (pixels)")
plt.xlabel("frame")
plt.legend()
for p in range(observations.num_people):
    seen = fit.seen[p]
    error = np.median(fit.keypoint_error[p, seen])
    print(f"person {p}: seen on {seen.mean():.0%} of frames, median error {error:.1f} px")

## The clip

Seen from above: where each person's root goes over the clip. The clip's world is levelled (the floor under the lowest points is z = 0) and centred on the people at the first frame. One camera sees no depth, so how far each person moves towards or away from it is the least sure part.

In [ ]:
plt.figure(figsize=(6, 6))
for p in range(clip.num_people):
    path = clip.root_translation[p]
    plt.plot(*path[:, :2].T, color=COLOURS[p % len(COLOURS)] / 255, label=f"person {p}")
    plt.scatter(*path[0, :2], color=COLOURS[p % len(COLOURS)] / 255, marker="o")
plt.scatter(*clip.camera_to_world[:2, 3], marker="^", color="k", label="camera")
plt.axis("equal")
plt.xlabel("x (m)")
plt.ylabel("y (m)")
plt.legend();

## The avatar performing it

The exported avatar, one copy per person, played by the reference runtime from the video's camera: the video, the two over each other, and the avatar alone.

In [ ]:
runtime = AvatarRuntime(read_avatar(exported_path(config))).to(device)
background = torch.ones(3, device=device)
camera = clip_camera(clip, fit.camera().scaled(0.4).to(device))
renders = list(render_clip(runtime, clip, camera, background, frames=shown.tolist()))
tiles = []
for f, render in zip(shown, renders, strict=True):
    video = np.array(Image.fromarray(frames[f]).resize(render.shape[1::-1]))
    tiles.append(np.concatenate([video, (0.5 * video + 0.5 * render).astype(np.uint8), render]))
plt.figure(figsize=(20, 14))
plt.imshow(np.concatenate(tiles, axis=1))
plt.axis("off");

From the side: what the one camera could not see, how far forwards and back the body leans and moves.

In [ ]:
# The root sits at the hips: look at the middle of where the people's hips go.
target = torch.tensor(clip.root_translation.reshape(-1, 3).mean(axis=0), device=device).float()
side = camera.looking_at(target + torch.tensor([3.5, -1.0, 0.3], device=device), target)
renders = list(render_clip(runtime, clip, side, background, frames=shown.tolist()))
plt.figure(figsize=(20, 5))
plt.imshow(np.concatenate(renders, axis=1))
plt.axis("off");

The whole clip as a video, the source beside the avatar, written to `outputs/motion/<name>/preview.mp4`. `PREVIEW_SECONDS` limits it, for long videos.

In [ ]:
PREVIEW_SECONDS = 30
count = min(len(frames), int(PREVIEW_SECONDS * clip.fps))
preview = motion_path(config, NAME, "clip").with_name("preview.mp4")
renders = render_clip(runtime, clip, camera, background, frames=range(count))
write_video(
    preview,
    (
        np.concatenate([np.array(Image.fromarray(frame).resize(r.shape[1::-1])), r], axis=1)
        for frame, r in zip(frames[:count], renders, strict=True)
    ),
    clip.fps,
)
Video(str(preview), embed=True, width=900)

## Against the full fit

Where notebook 06 has fitted the same video (results named `<video>`), the two fits' keypoint errors side by side: what the masks and silhouettes add.

In [ ]:
full = motion_path(config, VIDEO.stem, "fit")
if full.exists():
    masked = MotionFit.load(full)
    plt.figure(figsize=(12, 3))
    plt.plot(fit.frame_indices, fit.keypoint_error[0], label="keypoints only")
    plt.plot(masked.frame_indices, masked.keypoint_error[0], label="with masks and silhouettes")
    plt.ylabel("keypoint error (pixels)")
    plt.xlabel("frame")
    plt.legend()
else:
    print(f"no full fit of {VIDEO.name}: run notebook 06 on it to compare")